In [8]:
import os, glob, json
import unicodedata
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# DATA_PATH = '/kaggle/input/tashkeel-dataset/dataset/'
DATA_PATH = 'dataset/'
TRAIN_FILE = os.path.join(DATA_PATH, 'train.txt')
VAL_FILE = os.path.join(DATA_PATH, 'val.txt')
OUTPUT_MODEL_PATH = '/kaggle/working/bilstm_diac_pytorch_with_der.pt'

# Hyperparameters
MAXLEN = 500
EMBED_DIM = 25
LSTM_UNITS = 256
FF_UNITS = 512
DROPOUT = 0.5
BATCH_SIZE = 256
EPOCHS = 50
PLACEHOLDER = '<NONAR>'
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'
SPACE_TOKEN = '<SPACE>'


Using device: cuda


In [9]:
# Check if a character is a combining diacritic
def is_combining(ch):
    return unicodedata.category(ch) == 'Mn'

# Split a string into (base_char, diacritics) pairs
def split_char_diacritic_pairs(sentence):
    pairs = []
    base = None
    diacs = ''
    for ch in sentence:
        if is_combining(ch):
            if base is None:
                base = '<UNK_BASE>'
            diacs += ch
        else:
            if base is not None:
                pairs.append((base, diacs))
            base = ch
            diacs = ''
    # Avoid losing the last pair
    if base is not None:
        pairs.append((base, diacs))
    return pairs

# Check if a character is an Arabic letter
def is_arabic_letter(ch):
    # Check if ch is a single character
    if not isinstance(ch, str) or len(ch) != 1:
        return False
    # Converts the character into its Unicode code point
    code = ord(ch)
    return (
        (0x0600 <= code <= 0x06FF) or
        (0x0750 <= code <= 0x077F) or
        (0x08A0 <= code <= 0x08FF) or
        (0xFB50 <= code <= 0xFDFF) or
        (0xFE70 <= code <= 0xFEFF)
    )

import unicodedata

def is_gold_arabic_char(ch):
    # must be a single visible character
    if not isinstance(ch, str) or len(ch) != 1:
        return False

    # exclude tatweel explicitly
    if ch == "ـ":
        return False

    # must be a letter
    if not unicodedata.category(ch).startswith("L"):
        return False

    # Unicode name must indicate Arabic
    try:
        return "ARABIC" in unicodedata.name(ch)
    except ValueError:
        return False



# Transform (base_char, diacritics) pairs with placeholders
def placeholder_transform_pairs(pairs, placeholder=PLACEHOLDER):
    tokens = []
    labels = []
    for base, d in pairs:
        # Handle <UNK_BASE>
        if isinstance(base, str) and base.startswith('<') and base.endswith('>'):
            tokens.append(base)
            labels.append('')
        # Handle space
        elif base.isspace():
            tokens.append(SPACE_TOKEN)
            labels.append('')
        # Handle Arabic letters and Tatweel
        elif is_arabic_letter(base) or base == 'ـ':
            tokens.append(base)
            labels.append(d)
        # Handle other non-Arabic characters as Numbers
        else:
            tokens.append(placeholder)
            labels.append('')
    return tokens, labels


In [10]:
class BiLSTM_Diac(nn.Module):
    def __init__(self, vocab_size, emb_dim, lstm_units, ff_units, num_labels, pad_idx=0, dropout=0.5):
        super().__init__()
        # Map input character indices to dense embeddings, ensuring padding_idx is not updated during training
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # BiDirectional LSTM layers, with dropout for regularization (Avoid overfitting & memorization by randomly dropping units)
        self.bilstm1 = nn.LSTM(emb_dim, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(dropout)
        self.bilstm2 = nn.LSTM(2*lstm_units, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(dropout)

        # Feedforward layers to mix LSTM outputs to diacritic label logits
        self.ff1 = nn.Linear(2*lstm_units, ff_units)
        self.ff2 = nn.Linear(ff_units, ff_units)
        self.out = nn.Linear(ff_units, num_labels)
        self.relu = nn.ReLU()
    # Forward pass
    def forward(self, x, lengths=None):
        emb = self.embedding(x)
        if lengths is not None:
            # Pack padded sequence for efficient processing by LSTM
            packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out1, _ = self.bilstm1(packed)
            # Unpack the sequence back to padded form
            out1, _ = nn.utils.rnn.pad_packed_sequence(packed_out1, batch_first=True)
        else:
            # Directly pass embeddings through the first BiLSTM layer, Used in validation when all sequences are of same length
            out1, _ = self.bilstm1(emb)
        out1 = self.dropout1(out1)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(out1, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out2, _ = self.bilstm2(packed)
            out2, _ = nn.utils.rnn.pad_packed_sequence(packed_out2, batch_first=True)
        else:
            out2, _ = self.bilstm2(out1)
        out2 = self.dropout2(out2)
        # Feedforward layers with ReLU activations
        ff = self.relu(self.ff1(out2))
        ff = self.relu(self.ff2(ff))
        logits = self.out(ff)
        return logits

In [11]:
# =========================
# Inference → CSV + RAW/DIAC printing (CHUNKED, SAFE)
# =========================

import os
import csv
import pickle
import unicodedata
import torch

# -------------------------
# Load external label mapping (AUTHORITATIVE)
# -------------------------
PICKLE_LABEL_MAP_PATH = "diacritic2id.pickle"

with open(PICKLE_LABEL_MAP_PATH, "rb") as f:
    external_diac2label = pickle.load(f)

external_label2diac = {v: k for k, v in external_diac2label.items()}

print("External diacritic → label mapping:")
for k, v in external_diac2label.items():
    print(f"'{k}': {v}")

# -------------------------
# Load trained model
# -------------------------
MODEL_PATH = "Output/bilstm_Train_with_Val.pt"
ckpt = torch.load(MODEL_PATH, map_location=device)

char2idx = ckpt["char2idx"]
diac2idx = ckpt["diac2idx"]
idx2diac = {v: k for k, v in diac2idx.items()}

pad_idx = char2idx[PAD_TOKEN]

model = BiLSTM_Diac(
    vocab_size=len(char2idx),
    emb_dim=EMBED_DIM,
    lstm_units=LSTM_UNITS,
    ff_units=FF_UNITS,
    num_labels=len(diac2idx),
    pad_idx=pad_idx,
    dropout=DROPOUT,
).to(device)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# -------------------------
# Helpers
# -------------------------
def prepare_inference_raw_line(raw_line, placeholder=PLACEHOLDER):
    tokens = []
    original_nonar = []

    for ch in raw_line:
        if is_arabic_letter(ch) or ch == "ـ":
            tokens.append(ch)
        elif ch.isspace():
            tokens.append(SPACE_TOKEN)
            original_nonar.append((len(tokens) - 1, ch))
        else:
            tokens.append(placeholder)
            original_nonar.append((len(tokens) - 1, ch))

    return tokens, original_nonar


def training_label_to_external_label(train_label_id):
    diac = idx2diac.get(int(train_label_id), "")
    if diac in ("<NONE>", "<PAD_LABEL>", "<OTHER>", None):
        diac = ""
    return external_diac2label.get(diac, external_diac2label.get("", 0))


def reconstruct_diacritized_line(tokens, pred_ids, original_nonar):
    out = []
    nonar_map = {pos: ch for pos, ch in original_nonar}

    for i, (tok, pid) in enumerate(zip(tokens, pred_ids)):
        if i in nonar_map:
            out.append(nonar_map[i])
            continue

        if tok == SPACE_TOKEN:
            out.append(" ")
            continue

        diac = idx2diac.get(int(pid), "")
        if diac in ("<NONE>", "<PAD_LABEL>", "<OTHER>"):
            diac = ""

        out.append(unicodedata.normalize("NFC", tok + diac))

    return "".join(out)


def is_gold_arabic_char(ch):
    if not isinstance(ch, str) or len(ch) != 1:
        return False
    if ch == "ـ":
        return False
    if not unicodedata.category(ch).startswith("L"):
        return False
    try:
        return "ARABIC" in unicodedata.name(ch)
    except ValueError:
        return False

# -------------------------
# CHUNKED inference (CORE FIX)
# -------------------------
WINDOW = MAXLEN           # MUST match training
OVERLAP = 50
STRIDE = WINDOW - OVERLAP - 2

def infer_tokens_chunked(tokens):
    all_pred_ids = []
    pos = 0

    while pos < len(tokens):
        chunk = tokens[pos : pos + STRIDE]

        seq_mod = [SOS_TOKEN] + chunk + [EOS_TOKEN]
        x_ids = [char2idx.get(t, char2idx[UNK_TOKEN]) for t in seq_mod]
        x_ids += [char2idx[PAD_TOKEN]] * (WINDOW - len(x_ids))

        x_tensor = torch.tensor([x_ids], dtype=torch.long).to(device)
        lengths = torch.tensor([len(seq_mod)], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(x_tensor, lengths)
            pred = torch.argmax(logits, dim=-1)[0]

        pred = pred[1 : 1 + len(chunk)]
        all_pred_ids.extend(pred.cpu().tolist())

        pos += STRIDE

    return all_pred_ids[:len(tokens)]

# -------------------------
# Inference → CSV + printing
# -------------------------
def infer_text_to_csv(raw_text, output_csv_path):
    rows = []
    global_id = 0

    for raw_line in raw_text.splitlines():
        if not raw_line.strip():
            print()
            continue

        tokens, original_nonar = prepare_inference_raw_line(raw_line)
        pred_ids = infer_tokens_chunked(tokens)

        diac_line = reconstruct_diacritized_line(tokens, pred_ids, original_nonar)
        print("RAW:  ", raw_line)
        print("DIAC: ", diac_line)
        print()

        for tok, pid in zip(tokens, pred_ids):
            if not is_gold_arabic_char(tok):
                continue

            ext_label = training_label_to_external_label(pid)
            rows.append((global_id, ext_label))
            global_id += 1

    with open(output_csv_path, "w", newline="", encoding="utf8") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "label"])
        writer.writerows(rows)

    print(f"Saved CSV predictions to: {output_csv_path}")

# -------------------------
# Run
# -------------------------
test_file_path = "dataset_no_diacritics.txt"

with open(test_file_path, "r", encoding="utf8") as f:
    raw_text = f.read()

infer_text_to_csv(raw_text, "predictions.csv")


External diacritic → label mapping:
'َ': 0
'ً': 1
'ُ': 2
'ٌ': 3
'ِ': 4
'ٍ': 5
'ْ': 6
'ّ': 7
'َّ': 8
'ًّ': 9
'ُّ': 10
'ٌّ': 11
'ِّ': 12
'ٍّ': 13
'': 14


C:\Users\EAST ASIA\AppData\Local\Temp\ipykernel_20704\2329752209.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MODEL_PATH, map_location=device)


RAW:   ما أحسن ما شكا أمره بين أضعاف مدحه
DIAC:  مَا أَحْسَنَ مَا شَكَا أَمَرَهُ بَيْنَ أَضْعَافِ مَدْحِهِ

RAW:   فقد نجا كونوا علماء وجالسوا العلماء
DIAC:  فَقَدْ نَجَا كُوِّنُوا عُلَمَاءَ وَجَالَسُوا الْعُلَمَاءَ

RAW:   وقال بزرجمهر من العلم أن لا تحتقر شيئا من العلم
DIAC:  وَقَالَ بِزُرْجَمْهِرٍ مِنْ الْعِلْمِ أَنْ لَا تَحْتَقِرَ شَيْئًا مِنْ الْعِلْمِ

RAW:   فينبغي لمن استدل بفطرته على استحسان الفضائل
DIAC:  فَيَنْبَغِي لِمَنْ اسْتَدَلَّ بِفِطْرَتِهِ عَلَى اسْتِحْسَانِ الْفَضَائِلِ

RAW:   كدر معروفا امتنان
DIAC:  كَدَرَ مَعْرُوفًا امْتِنَانٌ

RAW:   وفي الأذكار عن بعضهم : يسن لمن لم يتمكن منها لحدث أو شغل أو نحوه أن يقول ذلك أربعا ، قال المصنف : إنه لا بأس به .
DIAC:  وَفِي الْأَذْكَارِ عَنْ بَعْضِهِمْ : يُسَنُّ لِمَنْ لَمْ يَتَمَكَّنْ مِنْهَا لِحَدَثٍ أَوْ شُغْلٍ أَوْ نَحْوِهِ أَنْ يَقُولَ ذَلِكَ أَرْبَعًا ، قَالَ الْمُصَنِّفُ : إنَّهُ لَا بَأْسَ بِهِ .

RAW:   ناشدت منظمة الأغذية والزراعة الجهات المانحة والمتبرعين لجمع أكثر من 40 مليون دولار على وجه السرعة لتأمين البذار والأس

In [12]:
def strip_diacritics(word: str) -> str:
    # remove all combining marks
    return ''.join(ch for ch in word if not is_combining(ch))

def strip_diacritics_sentence(sent: str) -> str:
    return " ".join(strip_diacritics(w) for w in sent.split())

train_path = "dataset/train.txt"   # adjust if your path is different

def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

train_lines = read_lines("dataset/train.txt")
val_lines   = read_lines("dataset/val.txt")   # gold dev set

print("Train sentences:", len(train_lines))
print("Val sentences:  ", len(val_lines))

Train sentences: 50000
Val sentences:   2500


In [14]:
# build lexicon and unigram frequencies from train.txt
from collections import Counter, defaultdict
from typing import List

def build_lexicon_and_unigrams(lines: List[str]):
    """
    Build:
      - lexicon: base (undiacritized) word -> Counter of diacritized forms
      - word_freq: unigram counts of diacritized words
    """
    lexicon = defaultdict(Counter)
    word_freq = Counter()
    
    for line in lines:
        for word in line.split():
            # count the fully diacritized word itself
            word_freq[word] += 1
            
            # map base form to diacritized variant
            base = strip_diacritics(word)
            if base:  # we keep base==word as well (digits, punctuation, etc.)
                lexicon[base][word] += 1
                
    return lexicon, word_freq

lexicon, word_freq = build_lexicon_and_unigrams(train_lines)

print(f"Number of base forms in lexicon: {len(lexicon)}")
some_base = next(iter(lexicon))
print("Example base form:", some_base)
print("Top variants:", lexicon[some_base].most_common(5))

# Input 
# lines = 
# [
#     "كَتَبَ الطَّالِبُ الدَّرسَ",
#     "قَرَأَ الطَّالِبُ الكِتابَ",
# ]

# OutPut
# lexicon
# {
#     "كتب": Counter(
#     {
#         "كَتَبَ": 5,
#         "كُتُبٌ": 2
#     }),
#     "الطالب": Counter(
#     {
#         "الطَّالِبُ": 8,
#         "الطَّالِبَ": 3,
#         "الطَّالِبِ": 1
#     })
# }

# word_freq
# {
#     "كَتَبَ": 5,
#     "كُتُبٌ": 2,
#     "الطَّالِبُ": 8,
#     "الطَّالِبَ": 3,
#     "الطَّالِبِ": 1
# }

Number of base forms in lexicon: 116078
Example base form: ولو
Top variants: [('وَلَوْ', 8551), ('وَلَوِ', 1)]


In [15]:
# (optional): save lexicon
import pickle

with open("lexicon.pkl", "wb") as f:
    pickle.dump(lexicon, f)

In [16]:
# unigram post-processing

def postprocess_word(predicted_word: str, lexicon) -> str:
    """
    Apply unigram post-processing to a single predicted word.
    - If base form was seen in training and predicted form wasn't,
      replace with most frequent diacritized form for that base.
    """
    base = strip_diacritics(predicted_word)
    
    # base never seen in training → cannot correct
    if not base or base not in lexicon:
        return predicted_word
    
    candidates = lexicon[base]
    
    # predicted form already observed in training → trust the model
    if predicted_word in candidates:
        return predicted_word
    
    # otherwise replace with the most frequent diacritized form
    most_frequent_form = candidates.most_common(1)[0][0]
    return most_frequent_form

def postprocess_sentence(predicted_sentence: str, lexicon) -> str:
    """Apply unigram post-processing to every word in a predicted sentence."""
    words = predicted_sentence.split()
    corrected_words = [postprocess_word(w, lexicon) for w in words]
    return " ".join(corrected_words)


# Example
# Assume training data contained:
# lexicon["كتب"] =
# {
#     "كَتَبَ": 12,
#     "كُتُبٌ": 3
# }

# Case 1 — valid prediction
# postprocess_word("كَتَبَ", lexicon)
# → "كَتَبَ"      already seen, keep it

# Case 2 — invalid/unseen prediction
# postprocess_word("كُتِبَ", lexicon)
# → "كَتَبَ"      replace with most frequent form

# Case 3 — unseen word
# postprocess_word("مُعَالِجَةٌ", lexicon)
# → "مُعَالِجَةٌ"    unchanged

In [17]:
# -----------------------------------------------------------------------------
# Missing Cell: Inference with Unigram Smoothing -> CSV
# -----------------------------------------------------------------------------

def extract_labels_from_diacritized_string(diac_string):
    """
    Parses a diacritized string (smoothed) and extracts the label IDs 
    for every Gold Arabic character to match the CSV format.
    """
    labels = []
    i = 0
    while i < len(diac_string):
        ch = diac_string[i]
        
        # We only care about Gold Arabic letters (skipping numbers, punctuation, Tatweel)
        # matching the logic used in the original inference loop.
        if is_gold_arabic_char(ch):
            # Look ahead for combining diacritics
            diac_seq = ""
            j = i + 1
            while j < len(diac_string) and is_combining(diac_string[j]):
                diac_seq += diac_string[j]
                j += 1
            
            # Map the diacritic sequence to its ID (using the external map loaded earlier)
            # Default to 0 (empty) if not found, though lexicon words should generally match.
            label_id = external_diac2label.get(diac_seq, external_diac2label.get("", 0))
            labels.append(label_id)
            
            # Advance index to after the diacritics
            i = j
        else:
            i += 1
    return labels

def infer_smoothed_to_csv(raw_text, lexicon, output_csv_path="predictions_smoothed.csv"):
    rows = []
    global_id = 0
    
    print(f"Generating smoothed predictions to {output_csv_path}...")
    
    # Process line by line
    for raw_line in raw_text.splitlines():
        if not raw_line.strip():
            continue

        # 1. Base Model Inference
        # Get tokens and run the BiLSTM model
        tokens, original_nonar = prepare_inference_raw_line(raw_line)
        pred_ids = infer_tokens_chunked(tokens)
        
        # Reconstruct the sentence string from model outputs
        diac_line_model = reconstruct_diacritized_line(tokens, pred_ids, original_nonar)
        
        # 2. Unigram Smoothing
        # Correct the sentence using the Unigram Lexicon
        smoothed_line = postprocess_sentence(diac_line_model, lexicon)
        
        # 3. Extract Labels from the Smoothed String
        line_labels = extract_labels_from_diacritized_string(smoothed_line)
        
        # Add to global list with IDs
        for label in line_labels:
            rows.append((global_id, label))
            global_id += 1

    # Write final CSV
    with open(output_csv_path, "w", newline="", encoding="utf8") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "label"])
        writer.writerows(rows)
    
    print("Done.")

# --- Execute the Smoothed Inference ---
test_file_path = "dataset_no_diacritics.txt"

if os.path.exists(test_file_path):
    with open(test_file_path, "r", encoding="utf8") as f:
        raw_text_input = f.read()
    
    # Run the function (ensure 'lexicon' is already built from the previous cells)
    infer_smoothed_to_csv(raw_text_input, lexicon, "predictions_smoothed.csv")
else:
    print(f"File not found: {test_file_path}")

Generating smoothed predictions to predictions_smoothed.csv...
Done.
